#### Notebook for running React experiments

In [2]:
import sys, os
sys.path.append('..')
root  = '../root/'

In [19]:
import joblib
from util import summarize_react_trial, log_react_trial, save_agents
from agents import ReactReflectAgent, ReactAgent, ReflexionStrategy, ReactReflectMemAgent
from llm import AnyOpenAILLM
from prompts import reflect_prompt_memory, strategies_prompt
from fewshots import REFLECTIONS

import pprint as pp

#### Load the HotpotQA Sample

In [4]:
hotpot = joblib.load('../data/hotpot-qa-distractor-sample.joblib').reset_index(drop = True)

print(type(hotpot))
hotpot.columns
hotpot.iloc[0]

#Xd, ok, so they only tested on hard questions
hard_ones = hotpot[hotpot["level"] == "hard"]
# hard_ones[["Woman's Era and Naj are what kind of magazines?"]]
test_q = hard_ones[hard_ones["question"] == "Woman's Era and Naj are what kind of magazines?"]

test_q

<class 'pandas.core.frame.DataFrame'>


,id,question,answer,type,level,supporting_facts,context
4,5a78bc6b554299148911f979,Woman's Era and Naj are what kind of magazines?,fortnightly women interest magazine,comparison,hard,"{'title': ['Woman's Era', 'Naj'], 'sent_id': [...","{'title': ['Lifestyle trends and media', 'Chin..."


#### Define the Reflexion Strategy

In [4]:
print(ReflexionStrategy.__doc__)


    NONE: No reflection
    LAST_ATTEMPT: Use last reasoning trace in context 
    REFLEXION: Apply reflexion to the next reasoning trace 
    LAST_ATTEMPT_AND_REFLEXION: Use last reasoning trace in context and apply reflexion to the next reasoning trace 
    


In [ ]:
from langchain.prompts import PromptTemplate

POST_REFLECTION_INCORRECT = """You are a highly capable, advanced reasoning agent that is capable of observing the thought process of another agent and guiding to find the best way to solve its problems. 
You will be given a previous reasoning trial from another agent that was given access to an Docstore API environment and a question to answer, as well as a set of reflections that you gave it, as it was struggling to find the correct answer. 
You will get to view not only all the actions it took, but also the reflections it used to help guide its decision making in taking such actions.The agent was unsuccessful in coming up with the correct answer, even with the help of your reflections. 
You will now get to view and correct answer, and with it, you will consider why your reflections failed to help the model, and give concrete, specific, but brief reasons for why the model failed.

**Output only a bullet list of reasons to take. Do not include introductions, summaries, transitions, or any narrative framing. Produce only the reasons themselves.**

Previous trial:
Question: {question}{scratchpad}

Correct Answer : {correct_answer}
"""

GENERATE_STRATEGIES_PROMPT = """You are a highly capable, advanced reasoning agent that is capable of observing the thought process of another agent and condensing it down to its most crucial-first principles. 
You are analyzing the reasoning traces of another agent that given access to an Docstore API environment, a question to answer, and a set of reflections meant to guide it. Unfortunately the agetn failed to correctly answer the question.
You will also be given a list of potential reasons for why the agent failed to arrive at the correct answer. Your job is to analyze these reasons and to come up with unconventional, novel, actionable, problem solving strategies that could've been used to guide the model. 
These general purpose strategies will be used by that same agent to solve a completely different question that involving searching for through a Docstore API to find an answer, so make sure that they target the core problem of searching through information to find an answer, 
and not the specifics of any certain question. 

**Output only a bullet list of strategies. Do not include introductions, summaries, transitions, or any narrative framing. Produce only the strategies themselves.**


Reasoning Traces:
Question: {question}{scratchpad}

Explanations for failure:
{explanations}
"""


post_mortem_prompt = PromptTemplate(
                        input_variables=["question", "scratchpad", "correct_answer"],
                        template = POST_REFLECTION_INCORRECT,
                        )

strategies_prompt = PromptTemplate(
                        input_variables=["question", "scratchpad", "explanations"],
                        template = GENERATE_STRATEGIES_PROMPT,
                        )

In [29]:
SCRATCHPAD = """
Reflections:
- I assumed that Woman's Era and Naj were the same kind of magazine without searching Naj. I should have searched Naj first and then compared the two magazines to see if they were the same kind of magazine before answering.
- I assumed that Woman's Era and Naj were the same kind of magazine without comparing the two. I should have compared the two magazines to see if they were the same kind of magazine before attempting to answer.
- I assumed that Woman's Era and Naj were the same type of magazine because they were both fortnightly magazines, but I should have looked more closely at the content of each magazine before answering. I should have searched both magazines and compared the topics they cover, such as fashion, cookery, movie and book review, health, relationships, beauty, lifestyle, travel and technology, to make sure they are the same type of magazine before answering.
- I assumed that Woman's Era and Naj were the same kind of magazine because they both covered topics such as fashion, lifestyle, beauty, health, relationships, travel, and technology. However, I should have taken into account that Woman's Era is an Indian magazine and Naj is a Polish magazine, and that the content of the two magazines may differ due to cultural differences. To mitigate this failure, I should research the content of both magazines more thoroughly before attempting to answer the question.
Thought 1: I need to search Woman's Era and Naj, compare their content, and find what kind of magazines they are.
Action 1: Search[Woman's Era]
Observation 1: Woman's Era is an Indian fortnightly women interest magazine, published in English. It was started in 1973 by Vishwanath under his publishing house, the Delhi Press. The magazine is owned by the Delhi Press. Divesh Nath has been the managing editor of the magazine since 2002.
Woman's Era covers diverse topics including fashion, cookery, movie and book review, health, relationships, beauty, lifestyle, travel and technology, with comments on socialites and current events. It includes poems and short stories. It is the second most popular women's magazine after Femina, with an All India Index of 80 as surveyed by the Indian Readership Survey (IRS).
Thought 2: Woman's Era covers topics such as fashion, cookery, movie and book review, health, relationships, beauty, lifestyle, travel and technology. I need to search Naj next and compare its content with Woman's Era.
Action 2: Search[Naj]
Observation 2: Naj is a Polish language fortnightly lifestyle and women's magazine published in Warsaw, Poland.
Thought 3: Naj is a Polish language fortnightly lifestyle and women's magazine. I need to compare its content with Woman's Era to see if they are the same kind of magazine.
Action 3: Lookup[compare]
Observation 3: No Results
Thought 4: I need to compare the topics Naj and Woman's Era cover, such as fashion, cookery, movie and book review, health, relationships, beauty, lifestyle, travel and technology, to make sure they are the same type of magazine.
Action 4: Lookup[topics]
Observation 4: No Results
Thought 5: I need to compare the content of Naj and Woman's Era to see if they are the same kind of magazine.
Action 5: Lookup[content]
Observation 5: No Results
Thought 6: I need to compare the topics Naj and Woman's Era cover, such as fashion, cookery, movie and book review, health, relationships, beauty, lifestyle, travel and technology, to make sure they are the same type of magazine.
Action 6: Lookup[fashion, cookery, movie and book review, health, relationships, beauty, lifestyle, travel and technology]
Observation 6: No Results
Correct answer: fortnightly women interest magazine
"""

In [38]:
#TODO: Sanity check on few basic examples to verify that everything works the way its supposed to, and if it does, create a seperate 
#agent/llm class just for the reflector, and pass that into each react agent

#Code that they used
def format_step(step: str) -> str:
    return step.strip('\n').strip().replace('\n', '')

#Because its generating multiple ideas, we want a higher temperature
reflector = AnyOpenAILLM(
    temperature=.25,
    max_tokens=100,
    model_name = "gpt-3.5-turbo",
    # model_name = "gpt-4-turbo",
    model_kwargs={"stop": "\n"},
    openai_api_key=os.environ['OPENAI_API_KEY'])

#Scratchpad might not be the right term because its the scratchpad + the actual reflections given to it
resolutions = []
for i in range(3):
    resolutions.append(reflector(post_mortem_prompt.format(question = test_q["question"], correct_answer = test_q["answer"], scratchpad = SCRATCHPAD)))

pp.pprint(resolutions)

strategies = []
for resolution in resolutions:
    strategies.append(reflector(strategies_prompt.format(question = test_q["question"], explanations = resolutions, scratchpad = SCRATCHPAD)))

strategies

["- The model failed to directly compare the content of Woman's Era and Naj to "
 'determine the type of magazines they are.',
 "- The model failed to directly compare the content of Woman's Era and Naj to "
 'determine the type of magazines they are.',
 "- The model failed to directly compare the content of Woman's Era and Naj to "
 'determine the type of magazines they are.']


["- Implement a content-based similarity analysis to compare the topics covered in Woman's Era and Naj to determine if they are the same type of magazine.",
 "- Implement a content-based similarity algorithm to compare the topics covered in Woman's Era and Naj to determine if they are the same type of magazine.",
 "- Implement a content-based similarity algorithm to compare the topics covered in Woman's Era and Naj to determine if they are the same type of magazine."]

In [ ]:



test_agent = ReactReflectMemAgent(question=test_q["question"], key = test_q["answer"], reflect_prompt=reflect_prompt_memory, strategy_memory=[])
for i in range(5):
    test_agent.run()

In [ ]:

#Basically, the way this currently works inside my head, 

#n trials loop -> we don't want it structured this way at all. We want it to run the n trials question after question
#after that, run the



#### Initialize a React Agent for each question

In [ ]:
agent_cls = ReactReflectAgent if strategy != ReflexionStrategy.NONE else ReactAgent
agents = [agent_cls(row['question'], row['answer']) for _, row in hotpot.iterrows()]

#### Run `n` trials

In [ ]:
n = 5
trial = 0
log = ''

In [ ]:
for i in range(n):
    for agent in [a for a in agents if not a.is_correct()]:
        if strategy != ReflexionStrategy.NONE:
            agent.run(reflect_strategy = strategy)
        else:
            agent.run()
        print(f'Answer: {agent.key}')
    trial += 1
    log += log_react_trial(agents, trial)
    correct, incorrect, halted = summarize_react_trial(agents)
    print(f'Finished Trial {trial}, Correct: {len(correct)}, Incorrect: {len(incorrect)}, Halted: {len(halted)}')

#### Save the result log

In [ ]:
with open(os.path.join(root, 'ReAct', strategy.value, f'{len(agents)}_questions_{trial}_trials.txt'), 'w') as f:
    f.write(log)
save_agents(agents, os.path.join('ReAct', strategy.value, 'agents'))